In [53]:
import pickle
import numpy as np

from pathlib import Path

from tensorflow.keras.models import load_model, Model
from tensorflow.keras.layers import Input
from tensorflow.keras.preprocessing.sequence import pad_sequences

print("🚀 HelpDesk AI - Inference")

🚀 HelpDesk AI - Inference


In [54]:
PROJECT_ROOT = Path.cwd().parent

MODEL_DIR = PROJECT_ROOT / "saved_models"

In [55]:
with open(MODEL_DIR / "tokenizer.pkl", "rb") as f:
    tokenizer = pickle.load(f)

print("Vocabulary:", len(tokenizer.word_index))

Vocabulary: 11495


In [56]:
model = load_model(
    MODEL_DIR / "chatbot_model.keras"
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, 64)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, 64)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_embedding   │ (None, 64, 64)    │    735,744 │ encoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 64)        │          0 │ encoder_inputs[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_embedding   │ (None, 64, 64)    │    735,744 │ decoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 128),     │     98,816 │ encoder_embeddin… │
│                     │ (None, 128),      │            │ not_equal[0][0]   │
│                     │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, 64, 128), │     98,816 │ decoder_embeddin… │
│                     │ (None, 128),      │            │ encoder_lstm[0][… │
│                     │ (None, 128)]      │            │ encoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_output      │ (None, 64, 11496) │  1,482,984 │ decoder_lstm[0][… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 9,456,314 (36.07 MB)

 Trainable params: 3,152,104 (12.02 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 6,304,210 (24.05 MB)

In [57]:
encoder_inputs = model.input[0]

encoder_embedding = model.get_layer("encoder_embedding")
encoder_lstm = model.get_layer("encoder_lstm")

encoder_outputs, state_h, state_c = encoder_lstm(
    encoder_embedding(encoder_inputs)
)

encoder_model = Model(
    encoder_inputs,
    [state_h, state_c]
)

In [58]:
decoder_inputs = model.input[1]

decoder_embedding = model.get_layer("decoder_embedding")
decoder_lstm = model.get_layer("decoder_lstm")
decoder_dense = model.get_layer("decoder_output")

decoder_state_input_h = Input(shape=(128,))
decoder_state_input_c = Input(shape=(128,))

decoder_states_inputs = [
    decoder_state_input_h,
    decoder_state_input_c
]

decoder_embedding_output = decoder_embedding(decoder_inputs)

decoder_outputs, state_h, state_c = decoder_lstm(
    decoder_embedding_output,
    initial_state=decoder_states_inputs
)

decoder_states = [
    state_h,
    state_c
]

decoder_outputs = decoder_dense(
    decoder_outputs
)

decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs] + decoder_states
)

In [59]:
reverse_word_index = {
    index: word
    for word, index in tokenizer.word_index.items()
}

print(len(reverse_word_index))

11495


In [60]:
MAX_QUESTION_LENGTH = 64
MAX_RESPONSE_LENGTH = 20

def encode_input(sentence):

    sentence = sentence.lower()

    sequence = tokenizer.texts_to_sequences([sentence])

    sequence = pad_sequences(
        sequence,
        maxlen=MAX_QUESTION_LENGTH,
        padding="post",
        truncating="post"
    )

    states = encoder_model.predict(
        sequence,
        verbose=0
    )

    return states

In [61]:
def generate_response(sentence):

    states = encode_input(sentence)

    target_seq = np.zeros((1, 1))

    target_seq[0, 0] = tokenizer.word_index["sos"]

    decoded_sentence = ""

    while True:

        output_tokens, h, c = decoder_model.predict(
            [target_seq] + states,
            verbose=0
        )

        sampled_token_index = np.argmax(
            output_tokens[0, -1, :]
        )

        sampled_word = reverse_word_index.get(
            sampled_token_index,
            ""
        )

        # Skip SOS and unknown tokens
        if sampled_word in ["sos", ""]:
            target_seq = np.zeros((1, 1))
            target_seq[0, 0] = sampled_token_index
            states = [h, c]
            continue

        # Stop generation
        if sampled_word == "eos":
            break

        if len(decoded_sentence.split()) >= MAX_RESPONSE_LENGTH:
            break

        decoded_sentence += sampled_word + " "

        target_seq = np.zeros((1, 1))
        target_seq[0, 0] = sampled_token_index

        states = [h, c]

    return decoded_sentence.strip().capitalize()

In [62]:
print(generate_response(
    "How can I cancel my order?"
))

I m on your side your need to check the estimated delivery time, you can follow these steps 1. sign


In [63]:
print(generate_response(
    "I forgot my password"
))

I m on it! i understand that you re seeking assistance in obtaining a rebate for your money. it can


In [64]:
print(generate_response(
    "How do I track my shipment?"
))

I ll make it happen! i understand your need to check the bill from salutation client last name. let me


In [65]:
print(generate_response(
    "Can I get a refund?"
))

I m on it! i understand that you re looking for assistance in obtaining a refund for your money. i
